# **Laboratorio #5**

*Juan Diego Letona*
*20230285*

In [1]:
#preparamos el corpus para trabajar secuencias

from pathlib import Path
import re
import random

TXT_PATH = Path("don-quijote.txt")

if not TXT_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo {TXT_PATH}")

#leemos el archivo aunque venga con otra codificacion
def leer_texto(path):
    for encoding in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError("No se pudo leer el archivo")

texto = leer_texto(TXT_PATH)
texto = re.sub(r"\s+", " ", texto).strip()

#hacemos la segmentacion en oraciones
oraciones_raw = re.split(r"(?<=[.!?])\s+", texto)

#tokenizamos cada oracion en palabras sin quitar stopwords ni lematizar
patron_palabras = re.compile(
    r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:[-'][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*|\d+(?:[.,]\d+)*",
    re.UNICODE
)

oraciones_tokenizadas = [
    patron_palabras.findall(oracion)
    for oracion in oraciones_raw
]

oraciones_tokenizadas = [
    oracion
    for oracion in oraciones_tokenizadas
    if len(oracion) > 0
]

#revisamos que la carga y tokenizacion tengan sentido
print(f"Archivo leido {TXT_PATH}")
print(f"Total de caracteres {len(texto):,}")
print(f"Total de oraciones tokenizadas {len(oraciones_tokenizadas):,}")
print("Ejemplo de oracion tokenizada")
print(oraciones_tokenizadas[0][:40])

Archivo leido don-quijote.txt
Total de caracteres 2,107,992
Total de oraciones tokenizadas 9,577
Ejemplo de oracion tokenizada
['The', 'Project', 'Gutenberg', 'EBook', 'of', 'Don', 'Quijote', 'by', 'Miguel', 'de', 'Cervantes', 'Saavedra', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever']


In [2]:
#agregamos tokens especiales al inicio y al final de cada oracion

TOKEN_INICIO = "~~"
TOKEN_FIN = "~~"

oraciones_con_tokens = [
    [TOKEN_INICIO] + oracion + [TOKEN_FIN]
    for oracion in oraciones_tokenizadas
]

print(f"Total de oraciones con tokens especiales {len(oraciones_con_tokens):,}")
print("Ejemplo con tokens de inicio y fin")
print(oraciones_con_tokens[0][:45])

Total de oraciones con tokens especiales 9,577
Ejemplo con tokens de inicio y fin
['~~', 'The', 'Project', 'Gutenberg', 'EBook', 'of', 'Don', 'Quijote', 'by', 'Miguel', 'de', 'Cervantes', 'Saavedra', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '~~']


In [3]:
#dividimos las oraciones en entrenamiento validacion y prueba

SEED = 42
random.seed(SEED)

oraciones = oraciones_con_tokens.copy()
random.shuffle(oraciones)

total_oraciones = len(oraciones)

n_train = int(total_oraciones * 0.80)
n_val = int(total_oraciones * 0.10)

train = oraciones[:n_train]
val = oraciones[n_train:n_train + n_val]
test = oraciones[n_train + n_val:]

print(f"Total de oraciones {total_oraciones:,}")
print(f"Entrenamiento {len(train):,} ({len(train) / total_oraciones:.1%})")
print(f"Validacion {len(val):,} ({len(val) / total_oraciones:.1%})")
print(f"Prueba {len(test):,} ({len(test) / total_oraciones:.1%})")

Total de oraciones 9,577
Entrenamiento 7,661 (80.0%)
Validacion 957 (10.0%)
Prueba 959 (10.0%)


In [4]:
#calculamos el vocabulario de entrenamiento y las palabras no vistas en prueba

tokens_especiales = {TOKEN_INICIO, TOKEN_FIN}

palabras_train = [
    token
    for oracion in train
    for token in oracion
    if token not in tokens_especiales
]

palabras_test = [
    token
    for oracion in test
    for token in oracion
    if token not in tokens_especiales
]

vocabulario_train = set(palabras_train)

palabras_oov = [
    palabra
    for palabra in palabras_test
    if palabra not in vocabulario_train
]

vocab_size = len(vocabulario_train)
proporcion_oov = len(palabras_oov) / len(palabras_test)

#reportamos las metricas pedidas
print(f"Tamano del vocabulario de entrenamiento {vocab_size:,}")
print(f"Total de palabras en prueba {len(palabras_test):,}")
print(f"Palabras de prueba no vistas en entrenamiento {len(palabras_oov):,}")
print(f"Proporcion OOV en prueba {proporcion_oov:.4f}")
print(f"Proporcion OOV en prueba {proporcion_oov:.2%}")

print("\nEjemplos de palabras OOV")
print(sorted(set(palabras_oov))[:50])

Tamano del vocabulario de entrenamiento 22,511
Total de palabras en prueba 38,462
Palabras de prueba no vistas en entrenamiento 1,514
Proporcion OOV en prueba 0.0394
Proporcion OOV en prueba 3.94%

Ejemplos de palabras OOV
['10', '1500', '1887', '596', '60', '801', '809', '84116', 'ACADÉMICOS', 'Abre', 'Acomodada', 'Acudid', 'Aderezáronse', 'Alabo', 'Alcaná', 'Alcocer', 'Alejandría', 'Algo', 'Amohinábase', 'Amohinóse', 'Any', 'Apagaron', 'Apenino', 'Aprieta', 'Asilde', 'Augusta', 'Babie', 'Basilea', 'Bañares', 'Benalcázar', 'Bonita', 'Burguillos', 'BÉJAR', 'CONDE', 'Callaban', 'Cansábanse', 'Capilla', 'Carloto', 'Caño', 'Charní', 'Ciertos', 'City', 'Cogiéronle', 'Comendador', 'Compluto', 'Concilio', 'Confusas', 'Creating', 'Creyóle', 'Cuitada']


El vocabulario del conjunto de entrenamiento tiene 22,511 palabras distintas.
Sin embargo, en el conjunto de prueba aparecio una proporcion OOV de 3.94%.
Esto significa que algunas palabras de prueba nunca aparecieron durante el entrenamiento.

Este problema se relaciona directamente con la dispersion de datos, o data sparsity,
porque aunque el corpus tenga bastante texto, no todas las palabras ni todas las
combinaciones posibles aparecen suficientes veces. Como el modelo trabaja palabra
por palabra, necesita ejemplos previos para aprender buenas probabilidades.

Cuando aparece una palabra nueva o una secuencia rara en prueba, el modelo no tiene
suficiente evidencia para estimarla bien. Esa falta de ejemplos es la idea central
de la data sparsity. Mientras mas pequeno o mas variado sea el corpus, mas probable
es encontrar palabras nunca vistas en validacion o prueba.

In [5]:
#construimos el modelo unigrama con frecuencias relativas

from collections import Counter, defaultdict
import math

conteo_unigramas = Counter()

for oracion in train:
    for palabra in oracion:
        conteo_unigramas[palabra] += 1

total_palabras_train = sum(conteo_unigramas.values())

prob_unigrama = {}

for palabra, conteo in conteo_unigramas.items():
    prob_unigrama[palabra] = conteo / total_palabras_train

print(f"Total de tokens en entrenamiento {total_palabras_train:,}")
print(f"Tamano del vocabulario con tokens especiales {len(prob_unigrama):,}")
print("Ejemplos de probabilidades unigrama")
print(list(prob_unigrama.items())[:10])

Total de tokens en entrenamiento 321,957
Tamano del vocabulario con tokens especiales 22,512
Ejemplos de probabilidades unigrama
[('~~', 0.04759020614554117), ('Tiene', 9.318014517466618e-06), ('asimesmo', 8.386213065719957e-05), ('maheridas', 3.1060048391555393e-06), ('danzas', 1.2424019356622157e-05), ('así', 0.0023046555906534104), ('de', 0.04425125094344897), ('espadas', 5.280208226564417e-05), ('como', 0.00552868861369686), ('cascabel', 3.1060048391555393e-06)]


In [6]:
#construimos el modelo bigrama usando conteos de pares consecutivos

conteo_bigramas = defaultdict(int)
conteo_contextos_bigrama = defaultdict(int)

for oracion in train:
    for i in range(1, len(oracion)):
        palabra_anterior = oracion[i - 1]
        palabra_actual = oracion[i]

        conteo_bigramas[(palabra_anterior, palabra_actual)] += 1
        conteo_contextos_bigrama[palabra_anterior] += 1

prob_bigrama = {}

for bigrama, conteo in conteo_bigramas.items():
    palabra_anterior, palabra_actual = bigrama
    prob_bigrama[bigrama] = conteo / conteo_contextos_bigrama[palabra_anterior]

print(f"Total de bigramas distintos {len(prob_bigrama):,}")
print("Ejemplos de probabilidades bigrama")
print(list(prob_bigrama.items())[:10])

Total de bigramas distintos 133,693
Ejemplos de probabilidades bigrama
[(('~~', 'Tiene'), 0.0003915937867119175), (('Tiene', 'asimesmo'), 0.3333333333333333), (('asimesmo', 'maheridas'), 0.037037037037037035), (('maheridas', 'danzas'), 1.0), (('danzas', 'así'), 0.25), (('así', 'de'), 0.03369272237196765), (('de', 'espadas'), 0.0002105706464518846), (('espadas', 'como'), 0.058823529411764705), (('como', 'de'), 0.029213483146067417), (('de', 'cascabel'), 7.019021548396153e-05)]


In [7]:
#construimos el modelo trigrama usando conteos de tres palabras consecutivas

conteo_trigramas = defaultdict(int)
conteo_contextos_trigrama = defaultdict(int)

for oracion in train:
    for i in range(2, len(oracion)):
        palabra_1 = oracion[i - 2]
        palabra_2 = oracion[i - 1]
        palabra_actual = oracion[i]

        contexto = (palabra_1, palabra_2)
        trigrama = (palabra_1, palabra_2, palabra_actual)

        conteo_trigramas[trigrama] += 1
        conteo_contextos_trigrama[contexto] += 1

prob_trigrama = {}

for trigrama, conteo in conteo_trigramas.items():
    palabra_1, palabra_2, palabra_actual = trigrama
    contexto = (palabra_1, palabra_2)
    prob_trigrama[trigrama] = conteo / conteo_contextos_trigrama[contexto]

print(f"Total de trigramas distintos {len(prob_trigrama):,}")
print("Ejemplos de probabilidades trigrama")
print(list(prob_trigrama.items())[:10])

Total de trigramas distintos 243,173
Ejemplos de probabilidades trigrama
[(('~~', 'Tiene', 'asimesmo'), 0.3333333333333333), (('Tiene', 'asimesmo', 'maheridas'), 1.0), (('asimesmo', 'maheridas', 'danzas'), 1.0), (('maheridas', 'danzas', 'así'), 1.0), (('danzas', 'así', 'de'), 1.0), (('así', 'de', 'espadas'), 0.04), (('de', 'espadas', 'como'), 0.3333333333333333), (('espadas', 'como', 'de'), 1.0), (('como', 'de', 'cascabel'), 0.019230769230769232), (('de', 'cascabel', 'menudo'), 1.0)]


In [8]:
#calculamos la probabilidad de una oracion de validacion en los tres modelos

oracion_ejemplo = val[0]

def probabilidad_oracion_unigrama(oracion):
    probabilidad = 1.0

    for palabra in oracion:
        probabilidad *= prob_unigrama.get(palabra, 0)

    return probabilidad

def probabilidad_oracion_bigrama(oracion):
    probabilidad = 1.0

    for i in range(1, len(oracion)):
        bigrama = (oracion[i - 1], oracion[i])
        probabilidad *= prob_bigrama.get(bigrama, 0)

    return probabilidad

def probabilidad_oracion_trigrama(oracion):
    probabilidad = 1.0

    for i in range(2, len(oracion)):
        trigrama = (oracion[i - 2], oracion[i - 1], oracion[i])
        probabilidad *= prob_trigrama.get(trigrama, 0)

    return probabilidad

p_unigrama = probabilidad_oracion_unigrama(oracion_ejemplo)
p_bigrama = probabilidad_oracion_bigrama(oracion_ejemplo)
p_trigrama = probabilidad_oracion_trigrama(oracion_ejemplo)

print("Oracion de ejemplo del conjunto de validacion")
print(oracion_ejemplo)

print(f"\nProbabilidad con modelo unigrama {p_unigrama}")
print(f"Probabilidad con modelo bigrama {p_bigrama}")
print(f"Probabilidad con modelo trigrama {p_trigrama}")

if p_unigrama == 0 or p_bigrama == 0 or p_trigrama == 0:
    print("\nAlguna probabilidad dio 0 porque el modelo encontro palabras o n-gramas que no aparecieron en entrenamiento")

Oracion de ejemplo del conjunto de validacion
['~~', 'Verdad', 'es', 'que', 'cuando', 'él', 'tiene', 'hambre', 'parece', 'algo', 'tragón', 'porque', 'come', 'apriesa', 'y', 'masca', 'a', 'dos', 'carrillos', 'pero', 'la', 'limpieza', 'siempre', 'la', 'tiene', 'en', 'su', 'punto', 'y', 'en', 'el', 'tiempo', 'que', 'fue', 'gobernador', 'aprendió', 'a', 'comer', 'a', 'lo', 'melindroso', 'tanto', 'que', 'comía', 'con', 'tenedor', 'las', 'uvas', 'y', 'aun', 'los', 'granos', 'de', 'la', 'granada', '~~']

Probabilidad con modelo unigrama 0.0
Probabilidad con modelo bigrama 0.0
Probabilidad con modelo trigrama 0.0

Alguna probabilidad dio 0 porque el modelo encontro palabras o n-gramas que no aparecieron en entrenamiento


La regla de la cadena exacta dice que la probabilidad de una oracion depende de todo
el historial anterior de palabras. El problema es que eso casi no se puede estimar
en la practica, porque necesitariamos haber visto muchas veces cada secuencia completa
o cada contexto largo en el corpus de entrenamiento.

El supuesto de Markov simplifica esa idea. En vez de usar todo el historial anterior,
hacemos que el modelo solo mire una cantidad limitada de palabras previas. En un
unigrama no se mira ninguna palabra anterior. En un bigrama solo se mira la palabra
anterior. En un trigrama se miran las dos palabras anteriores.

Gracias a eso, la regla de la cadena deja de depender de contextos enormes y se
convierte en conteos mas pequenos que si podemos calcular con el corpus. El resultado
no es exacto, porque perdemos informacion del contexto largo, pero si es estimable.
Por eso los modelos n grama son una aproximacion practica de la regla de la cadena.

In [9]:
#implementamos suavizado de laplace y add k para bigramas

vocabulario_modelo = set(conteo_unigramas.keys())
UNK = "<UNK>"

#agregamos unk para poder manejar palabras que no aparecieron en entrenamiento
vocabulario_suavizado = vocabulario_modelo | {UNK}
V = len(vocabulario_suavizado)

def normalizar_palabra(palabra):
    if palabra in vocabulario_modelo:
        return palabra
    return UNK

def prob_bigrama_sin_suavizado(palabra_anterior, palabra_actual):
    palabra_anterior = normalizar_palabra(palabra_anterior)
    palabra_actual = normalizar_palabra(palabra_actual)

    return prob_bigrama.get((palabra_anterior, palabra_actual), 0)

def prob_bigrama_add_k(palabra_anterior, palabra_actual, k):
    palabra_anterior = normalizar_palabra(palabra_anterior)
    palabra_actual = normalizar_palabra(palabra_actual)

    conteo_par = conteo_bigramas.get((palabra_anterior, palabra_actual), 0)
    conteo_contexto = conteo_contextos_bigrama.get(palabra_anterior, 0)

    return (conteo_par + k) / (conteo_contexto + k * V)

def prob_bigrama_laplace(palabra_anterior, palabra_actual):
    return prob_bigrama_add_k(palabra_anterior, palabra_actual, k=1)

print(f"Tamano del vocabulario para suavizado {V:,}")
print("Modelos de suavizado listos")
print("Laplace usa k = 1")
print("Add k se puede probar con otros valores como 0.1 y 0.01")

Tamano del vocabulario para suavizado 22,513
Modelos de suavizado listos
Laplace usa k = 1
Add k se puede probar con otros valores como 0.1 y 0.01


In [10]:
#comparamos una misma oracion con y sin suavizado

def tiene_bigrama_no_visto(oracion):
    for i in range(1, len(oracion)):
        bigrama = (normalizar_palabra(oracion[i - 1]), normalizar_palabra(oracion[i]))

        if bigrama not in conteo_bigramas:
            return True

    return False

#buscamos una oracion de validacion que tenga al menos un bigrama no visto
oracion_suavizado = None

for oracion in val:
    if tiene_bigrama_no_visto(oracion):
        oracion_suavizado = oracion
        break

if oracion_suavizado is None:
    oracion_suavizado = val[0]

def prob_oracion_bigrama_sin_suavizado(oracion):
    probabilidad = 1.0

    for i in range(1, len(oracion)):
        probabilidad *= prob_bigrama_sin_suavizado(oracion[i - 1], oracion[i])

    return probabilidad

def prob_oracion_bigrama_add_k(oracion, k):
    probabilidad = 1.0

    for i in range(1, len(oracion)):
        probabilidad *= prob_bigrama_add_k(oracion[i - 1], oracion[i], k)

    return probabilidad

p_sin_suavizado = prob_oracion_bigrama_sin_suavizado(oracion_suavizado)
p_laplace = prob_oracion_bigrama_add_k(oracion_suavizado, k=1)
p_add_k_01 = prob_oracion_bigrama_add_k(oracion_suavizado, k=0.1)
p_add_k_001 = prob_oracion_bigrama_add_k(oracion_suavizado, k=0.01)

print("Oracion usada para comparar")
print(oracion_suavizado)

print(f"\nProbabilidad sin suavizado {p_sin_suavizado}")
print(f"Probabilidad con Laplace k 1 {p_laplace}")
print(f"Probabilidad con add k 0.1 {p_add_k_01}")
print(f"Probabilidad con add k 0.01 {p_add_k_001}")

print("\nBigramas no vistos en esta oracion")
for i in range(1, len(oracion_suavizado)):
    bigrama = (normalizar_palabra(oracion_suavizado[i - 1]), normalizar_palabra(oracion_suavizado[i]))

    if bigrama not in conteo_bigramas:
        print(bigrama)

Oracion usada para comparar
['~~', 'Verdad', 'es', 'que', 'cuando', 'él', 'tiene', 'hambre', 'parece', 'algo', 'tragón', 'porque', 'come', 'apriesa', 'y', 'masca', 'a', 'dos', 'carrillos', 'pero', 'la', 'limpieza', 'siempre', 'la', 'tiene', 'en', 'su', 'punto', 'y', 'en', 'el', 'tiempo', 'que', 'fue', 'gobernador', 'aprendió', 'a', 'comer', 'a', 'lo', 'melindroso', 'tanto', 'que', 'comía', 'con', 'tenedor', 'las', 'uvas', 'y', 'aun', 'los', 'granos', 'de', 'la', 'granada', '~~']

Probabilidad sin suavizado 0.0
Probabilidad con Laplace k 1 4.971294633838005e-199
Probabilidad con add k 0.1 7.733226600474352e-180
Probabilidad con add k 0.01 5.478794684924107e-172

Bigramas no vistos en esta oracion
('hambre', 'parece')
('algo', '<UNK>')
('<UNK>', 'porque')
('porque', 'come')
('come', 'apriesa')
('y', 'masca')
('masca', 'a')
('carrillos', 'pero')
('limpieza', 'siempre')
('fue', 'gobernador')
('gobernador', 'aprendió')
('aprendió', 'a')
('lo', 'melindroso')
('melindroso', 'tanto')
('comía',


El suavizado resuelve el problema de que una sola combinacion no vista haga que toda
la oracion tenga probabilidad cero. Sin suavizado, si un bigrama no aparecio en el
entrenamiento, su probabilidad es cero y al multiplicar por la regla de la cadena
toda la probabilidad de la oracion tambien se vuelve cero.

Laplace y add k corrigen esto sumando una cantidad pequena al conteo de todos los
bigramas posibles. Asi, los bigramas no vistos reciben una probabilidad pequena,
pero ya no quedan completamente descartados.

Esto tambien significa que los bigramas frecuentes pierden un poco de probabilidad.
No es que el modelo diga que ya no son importantes, sino que deja de confiar al cien
por ciento en lo que vio en entrenamiento. Esa probabilidad se reparte hacia casos
no vistos para aceptar que el corpus no contiene todas las combinaciones posibles.

Por eso el suavizado ayuda a evitar la sobreconfianza. Un modelo sin suavizado actua
como si todo lo que no vio fuera imposible. Un modelo suavizado reconoce que algo no
haya aparecido en entrenamiento no significa que sea imposible en validacion, prueba
o texto real.

In [11]:
#calculamos perplejidad de unigramas bigramas y trigramas con suavizado

import math
import pandas as pd

K_GENERAL = 1

def prob_unigrama_add_k(palabra, k=K_GENERAL):
    palabra = normalizar_palabra(palabra)

    conteo = conteo_unigramas.get(palabra, 0)

    return (conteo + k) / (total_palabras_train + k * V)

def prob_trigrama_add_k(palabra_1, palabra_2, palabra_actual, k=K_GENERAL):
    palabra_1 = normalizar_palabra(palabra_1)
    palabra_2 = normalizar_palabra(palabra_2)
    palabra_actual = normalizar_palabra(palabra_actual)

    contexto = (palabra_1, palabra_2)
    trigrama = (palabra_1, palabra_2, palabra_actual)

    conteo_tri = conteo_trigramas.get(trigrama, 0)
    conteo_contexto = conteo_contextos_trigrama.get(contexto, 0)

    return (conteo_tri + k) / (conteo_contexto + k * V)

def perplejidad_unigrama(oraciones, k=K_GENERAL):
    suma_log = 0
    total = 0

    for oracion in oraciones:
        for palabra in oracion:
            if palabra == TOKEN_INICIO:
                continue

            probabilidad = prob_unigrama_add_k(palabra, k)
            suma_log += math.log(probabilidad)
            total += 1

    return math.exp(-suma_log / total)

def perplejidad_bigrama(oraciones, k=K_GENERAL):
    suma_log = 0
    total = 0

    for oracion in oraciones:
        for i in range(1, len(oracion)):
            probabilidad = prob_bigrama_add_k(oracion[i - 1], oracion[i], k)
            suma_log += math.log(probabilidad)
            total += 1

    return math.exp(-suma_log / total)

def perplejidad_trigrama(oraciones, k=K_GENERAL):
    suma_log = 0
    total = 0

    for oracion in oraciones:
        for i in range(2, len(oracion)):
            probabilidad = prob_trigrama_add_k(oracion[i - 2], oracion[i - 1], oracion[i], k)
            suma_log += math.log(probabilidad)
            total += 1

    return math.exp(-suma_log / total)

ppl_unigrama_val = perplejidad_unigrama(val, k=K_GENERAL)
ppl_bigrama_val = perplejidad_bigrama(val, k=K_GENERAL)
ppl_trigrama_val = perplejidad_trigrama(val, k=K_GENERAL)

print(f"Perplejidad unigrama en validacion {ppl_unigrama_val:.4f}")
print(f"Perplejidad bigrama en validacion {ppl_bigrama_val:.4f}")
print(f"Perplejidad trigrama en validacion {ppl_trigrama_val:.4f}")

Perplejidad unigrama en validacion 968.7392
Perplejidad bigrama en validacion 3525.3908
Perplejidad trigrama en validacion 14106.9942
